In [ ]:
#STEP 1 — Install LangGraph
!pip install -q -U langgraph langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 13.0 MB/s eta 0:00:00


In [ ]:
#STEP 2 — Import Libraries and Create State
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# Define the state of our chatbot
class ChatbotState(TypedDict):
    user_message: str
    chatbot_response: str
    is_valid: bool

In [ ]:
#STEP 3 — Create the Chatbot Node
def chatbot_node(state: ChatbotState):

    message = state["user_message"]

    # Simple chatbot responses
    if "hello" in message.lower():
        answer = "Hello! How can I help you with your studies?"

    elif "python" in message.lower():
        answer = "Python is a high-level programming language used for many applications."

    elif "langgraph" in message.lower():
        answer = "LangGraph is a framework used to build stateful workflows for AI applications."

    elif "database" in message.lower():
        answer = "A database is used to store, organize and manage data."

    else:
        answer = f"You asked: {message}. I am your Study Assistant Chatbot."

    return {
        "chatbot_response": answer
    }

In [ ]:
#STEP 4 — Create Validation Node
def validation_node(state: ChatbotState):

    response = state["chatbot_response"]

    # Check whether chatbot generated a meaningful response
    valid = len(response.strip()) > 10

    return {
        "is_valid": valid
    }

In [ ]:
#STEP 5 — Create Routing Function
def chatbot_router(state: ChatbotState):

    if state["is_valid"]:
        return "valid"

    else:
        return "invalid"

In [ ]:
#STEP 6 — Build the LangGraph
builder = StateGraph(ChatbotState)


# Add chatbot node
builder.add_node("chatbot", chatbot_node)

# Add validation node
builder.add_node("validator", validation_node)


# Starting point
builder.add_edge(START, "chatbot")

# Chatbot goes to validator
builder.add_edge("chatbot", "validator")


# Conditional routing
builder.add_conditional_edges(
    "validator",
    chatbot_router,
    {
        "valid": END,
        "invalid": "chatbot"
    }
)


# Compile the graph
app = builder.compile()

In [ ]:
#STEP 7 — Run the Chatbot
user_question = "What is AI?"

result = app.invoke({
    "user_message": user_question
})

print("----- STUDY ASSISTANT CHATBOT -----")
print("User:", result["user_message"])
print("Bot:", result["chatbot_response"])
print("Valid Response:", result["is_valid"])

----- STUDY ASSISTANT CHATBOT -----
User: What is AI?
Bot: You asked: What is AI?. I am your Study Assistant Chatbot.
Valid Response: True
